In [12]:
!pip install deepface opencv-python tensorflow keras



In [13]:
from deepface import DeepFace
import cv2
import numpy as np
import json


In [14]:
EMOTION_TO_STRESS = {
    "angry": 0.9,
    "disgust": 0.8,
    "fear": 0.95,
    "sad": 0.7,
    "surprise": 0.5,
    "neutral": 0.2,
    "happy": 0.1
}




In [15]:
def extract_face_emotions(image_path: str):
    """
    Returns raw DeepFace emotion output
    """
    result = DeepFace.analyze(
        img_path=image_path,
        actions=["emotion"],
        enforce_detection=False
    )

    emotions = result[0]["emotion"]
    dominant_emotion = result[0]["dominant_emotion"]
    face_confidence = result[0].get("face_confidence", 0.0)

    return emotions, dominant_emotion, face_confidence



In [16]:
def compute_face_stress_score(emotion_probs: dict) -> float:
    """
    Weighted sum of emotion probabilities → normalized stress score
    """
    stress_score = 0.0

    for emotion, prob in emotion_probs.items():
        stress_score += (prob / 100.0) * EMOTION_TO_STRESS.get(emotion, 0.5)

    # Clamp for safety
    stress_score = max(0.0, min(1.0, stress_score))
    return round(stress_score, 3)



In [17]:
def predict_face_stress(image_path: str) -> dict:
    """
    Face-only inference.
    No fusion, no weighting, no decisions.

    Returns:
    {
        "face_stress_score": float (0–1),
        "dominant_emotion": str,
        "face_confidence": float
    }
    """
    emotion_probs, dominant_emotion, face_confidence = extract_face_emotions(image_path)
    face_stress_score = compute_face_stress_score(emotion_probs)

    return {
        "face_stress_score": face_stress_score,
        "dominant_emotion": dominant_emotion,
        "face_confidence": round(float(face_confidence), 3)
    }


In [18]:
with open("face_emotion_mapper.json", "w") as f:
    json.dump(EMOTION_TO_STRESS, f, indent=2)

print("Saved face_emotion_mapper.json")


Saved face_emotion_mapper.json


In [20]:
from google.colab import files

files.download("face_emotion_mapper.json")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>